# Local audit of the 2-D elastic LHS: OPT versus FD

This notebook deliberately performs **no propagation and no source calculation**. It audits equation (54), the assembled homogeneous LHS, and conventional three-point finite differences. Equation (55)/Γ is outside the scope of this test.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    candidates = haskey(ENV, "FLEXOPT_ROOT") ? [ENV["FLEXOPT_ROOT"]] : String[]
    directory = abspath(start)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(abspath.(expanduser.(candidates)))
        isfile(joinpath(candidate, "src", "flexOPT.jl")) && return candidate
    end
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
using Metal
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .commonBatchs, .flexOPT
using CairoMakie, LinearAlgebra, SparseArrays, Statistics, JLD2, Dates
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() backend Base.active_project()

## Explicit equation versus tensor equation

The explicit version writes each stress-divergence term separately. This prevents an implicit tensor contraction from hiding a swapped index or a misplaced Lamé coefficient.

In [ ]:
tensorEquation, _ = famousEquations("2DsismoTimeIsoHeteroSingleForce")
explicitEquation, _ = famousEquations("2DsismoTimeIsoHeteroSingleForceExplicit")
println("Tensor/Tullio LHS:")
display(tensorEquation.exprs)
println("Explicit component LHS:")
display(explicitEquation.exprs)
@assert explicitEquation.vars == tensorEquation.vars
@assert explicitEquation.fields == tensorEquation.fields

## Build tiny homogeneous operators

Only a 17×17 grid is assembled. `OPT3-explicit` and `OPT3-tensor` differ only in the PDE declaration. `convFD3` uses `orderBspace = orderBtime = -1`.

In [ ]:
n = 17
coordinates = collect(-(n ÷ 2):(n ÷ 2))
ρ0, λ0, μ0 = 1.0, 2.3, 1.7
baseParameters = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForceExplicit",
    "Δ" => (1.0, 1.0, 1.0),
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "recipe_backend" => backend,
)
function cached_audit_recipe(parameters, prefix)
    cache = Dict(k => v for (k, v) in parameters if k != "recipe_backend")
    cache["lhs_audit_version"] = 3 # hierarchical constrained Taylor inverse
    producer = config -> begin
        runtime = Dict{String,Any}(config)
        pop!(runtime, "hash_id", nothing); pop!(runtime, "lhs_audit_version", nothing)
        runtime["recipe_backend"] = backend
        makeOPTsemiSymbolic(runtime)
    end
    myProduceOrLoad(producer, cache, "semiSymbolic", prefix)
end
explicitRecipe = cached_audit_recipe(baseParameters, "elasticLHS_OPT3_explicit")
tensorParameters = copy(baseParameters)
tensorParameters["famousEquationType"] = "2DsismoTimeIsoHeteroSingleForce"
tensorRecipe = cached_audit_recipe(tensorParameters, "elasticLHS_OPT3_tensor")
supp0Parameters = copy(baseParameters)
supp0Parameters["supplementaryOrder"] = 0
supp0Recipe = cached_audit_recipe(supp0Parameters, "elasticLHS_OPT3_supp0_explicit")
convParameters = copy(baseParameters)
convParameters["orderBspace"] = -1; convParameters["orderBtime"] = -1
convParameters["supplementaryOrder"] = 0
convRecipe = cached_audit_recipe(convParameters, "elasticLHS_convFD3_explicit")
higherOrderRecipes = Dict{Int,Any}()
for pointsInSpace in (4, 5)
    parameters = copy(baseParameters)
    parameters["pointsInSpace"] = pointsInSpace
    offset = (pointsInSpace - 1) / 2
    parameters["fieldItpl"] = merge(baseParameters["fieldItpl"],
        (offsetSpace=offset,))
    parameters["materItpl"] = merge(baseParameters["materItpl"],
        (offsetSpace=offset,))
    higherOrderRecipes[pointsInSpace] = cached_audit_recipe(parameters,
        "elasticLHS_OPT$(pointsInSpace)_explicit")
end

In [ ]:
# Reproduce makeOPTsemiSymbolic's symbolic-analysis stage and expose bigα.
trialCharacteristics = (orderBtime=baseParameters["orderBtime"],
    orderBspace=baseParameters["orderBspace"],
    pointsInSpace=baseParameters["pointsInSpace"],
    pointsInTime=baseParameters["pointsInTime"], nuGeometryMode=:middle)
fieldTaylor = flexOPT.TaylorOptions(baseParameters["fieldItpl"],
    baseParameters["supplementaryOrder"])
materialTaylor = flexOPT.TaylorOptions(baseParameters["materItpl"],
    baseParameters["supplementaryOrder"])
analysisNumbers = flexOPT.numbersOfTheExpression(explicitEquation,
    trialCharacteristics, fieldTaylor, materialTaylor)
_, fieldOrders, fieldConfig, _, materialConfig = flexOPT.investigateDependencies(explicitEquation,
    analysisNumbers, trialCharacteristics, fieldTaylor, materialTaylor)
bigα, varM, materialDependencies = flexOPT.bigαFinder(
    explicitEquation, analysisNumbers, fieldOrders)
function alpha_table(bigα)
    [(equation=i, field=j, coefficient=string(alpha.node),
      field_derivative=Tuple(alpha.n) .- 1,
      material_derivative=Tuple(alpha.nᶜ) .- 1)
     for i in axes(bigα,1), j in axes(bigα,2) for alpha in bigα[i,j]]
end
bigAlphaTable = alpha_table(bigα)
mixedFieldTerms = filter(row -> row.field_derivative == (1,1,0) &&
    row.material_derivative == (0,0,0), bigAlphaTable)
display(mixedFieldTerms)
@assert Set(row.coefficient for row in mixedFieldTerms if row.equation==1 && row.field==2) ==
    Set(("-λ(x, y)", "-μ(x, y)"))
@assert Set(row.coefficient for row in mixedFieldTerms if row.equation==2 && row.field==1) ==
    Set(("-λ(x, y)", "-μ(x, y)"))

# Audit the Taylor inverse before WYYKK/windowContraction!.  A maps Taylor
# coefficients to stencil values; C maps stencil values back to coefficients.
pointsTaylor = vec(fieldConfig.availablePointsConfigurations[1])
μTaylor = vec(fieldConfig.availableμPoints[1])[1]
ordersTaylor = vec(collect(fieldConfig.multiOrdersIndices))
ATaylor = Matrix{Float64}(undef, length(ordersTaylor), length(pointsTaylor))
for (j, orderIndex) in enumerate(ordersTaylor), (k, point) in enumerate(pointsTaylor)
    powers = Tuple(orderIndex) .- 1
    distances = Float64.(point .- μTaylor)
    ATaylor[j,k] = prod(distances .^ powers) / prod(factorial.(powers))
end
CTaylor = explicitRecipe["recette"].Cˡη[:,:,1]
taylorOrderProjector = ATaylor * CTaylor
order_row(powers) = findfirst(index -> Tuple(index) .- 1 == powers, ordersTaylor)
selectedTaylorReproduction = Dict(powers => (
        diagonal=taylorOrderProjector[order_row(powers),order_row(powers)],
        row_error=norm(taylorOrderProjector[order_row(powers),:] -
            Matrix(I, size(taylorOrderProjector)...)[order_row(powers),:]),
    ) for powers in ((2,0,0),(0,2,0),(1,1,0),(0,0,2)))
taylorAudit = (number_of_nodes=size(ATaylor,2),
    number_of_Taylor_terms=size(ATaylor,1), rank=rank(ATaylor),
    selected=selectedTaylorReproduction)
display(taylorAudit)

In [ ]:
models = [fill(ρ0, n, n), fill(λ0, n, n), fill(μ0, n, n)]
function prepare_audit_operator(recipe, name; points_in_space=3)
    marching = recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching
    points = getModelPoints(models[1], points_in_space, marching)
    family = (models=models, modelPoints=points, Δ=(1.0, 1.0, 1.0), modelName=name)
    numops = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => recipe, "modelFam" => family,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => nothing, "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numops)
end
preparedExplicit = prepare_audit_operator(explicitRecipe, "OPT3 explicit")
preparedTensor = prepare_audit_operator(tensorRecipe, "OPT3 tensor")
preparedSupp0 = prepare_audit_operator(supp0Recipe, "OPT3 supplementaryOrder=0")
preparedConv = prepare_audit_operator(convRecipe, "convFD3 explicit")
preparedOPT4 = prepare_audit_operator(higherOrderRecipes[4], "OPT4 explicit";
    points_in_space=4)
preparedOPT5 = prepare_audit_operator(higherOrderRecipes[5], "OPT5 explicit";
    points_in_space=5)
tensorExplicitMatrixDifference = (
    A_unknown=norm(preparedExplicit.A_unknown - preparedTensor.A_unknown) / max(norm(preparedTensor.A_unknown), eps()),
    L_known=norm(preparedExplicit.L_known - preparedTensor.L_known) / max(norm(preparedTensor.L_known), eps()),
)
@show preparedExplicit.spaceShape preparedExplicit.timePointsUsedForOneStep
@show explicitRecipe["recette"].taylorInverseMode
@show tensorExplicitMatrixDifference

## Compare local coefficients with standard centered FD

Time-independent fields cancel the three-point temporal second derivative. The remaining local stencil is compared after one optimal scalar normalization, because a weak-form row may be multiplied by an arbitrary nonzero test-function normalization.

In [ ]:
center = CartesianIndex(cld(n, 2), cld(n, 2))
centerLinear = LinearIndices((n, n))[center]
function static_stencil(prepared, equation, point=center)
    nspace, nfield = prepared.NpointsSpace, prepared.NField
    row = equation + nfield * (LinearIndices(prepared.spaceShape)[point] - 1)
    result = Dict{Tuple{Int,Int,Int},Float64}()
    for matrix in (prepared.A_unknown, prepared.L_known)
        columns, values = findnz(vec(matrix[row, :]))
        for (column, value) in zip(columns, values)
            within = mod1(column, nspace * nfield)
            field = cld(within, nspace)
            q = CartesianIndices(prepared.spaceShape)[mod1(within, nspace)]
            offset = Tuple(q - point)
            key = (field, offset[1], offset[2])
            result[key] = get(result, key, 0.0) + Float64(value)
        end
    end
    filter!(pair -> abs(last(pair)) > 1e-13, result)
end
function fd_stencil(equation)
    a, b, c = λ0 + 2μ0, μ0, λ0 + μ0
    d = Dict{Tuple{Int,Int,Int},Float64}()
    if equation == 1
        d[(1,0,0)] = 2a + 2b
        d[(1,-1,0)] = d[(1,1,0)] = -a
        d[(1,0,-1)] = d[(1,0,1)] = -b
        for (dx,dz) in ((1,1),(-1,-1)) d[(2,dx,dz)] = -c/4 end
        for (dx,dz) in ((1,-1),(-1,1)) d[(2,dx,dz)] = c/4 end
    else
        d[(2,0,0)] = 2b + 2a
        d[(2,-1,0)] = d[(2,1,0)] = -b
        d[(2,0,-1)] = d[(2,0,1)] = -a
        for (dx,dz) in ((1,1),(-1,-1)) d[(1,dx,dz)] = -c/4 end
        for (dx,dz) in ((1,-1),(-1,1)) d[(1,dx,dz)] = c/4 end
    end
    d
end
function compare_stencils(observed, reference)
    keysAll = collect(union(keys(observed), keys(reference)))
    o = [get(observed, k, 0.0) for k in keysAll]
    r = [get(reference, k, 0.0) for k in keysAll]
    scale = dot(o, r) / max(dot(o, o), eps())
    (; relative_error=norm(scale .* o .- r) / norm(r), scale, keys=keysAll, observed=o, reference=r)
end
coefficientMethods = (OPT3_explicit=preparedExplicit,
    OPT3_tensor=preparedTensor, OPT3_supp0=preparedSupp0,
    OPT4=preparedOPT4, OPT5=preparedOPT5, convFD3=preparedConv)
coefficientComparison = map(coefficientMethods) do prepared
    (equation1=compare_stencils(static_stencil(prepared, 1), fd_stencil(1)),
     equation2=compare_stencils(static_stencil(prepared, 2), fd_stencil(2)))
end
display(map(x -> (equation1_error=x.equation1.relative_error, equation2_error=x.equation2.relative_error,
                  equation1_scale=x.equation1.scale, equation2_scale=x.equation2.scale), coefficientComparison))

In [ ]:
function stencil_matrix(stencil, equationField; radius=2)
    [get(stencil, (equationField, dx, dz), 0.0) for dx in -radius:radius, dz in -radius:radius]
end
methods = coefficientMethods
fig = Figure(size=(1900, 850))
for (column, (name, prepared)) in enumerate(pairs(methods)), equation in 1:2
    ax = Axis(fig[equation, column]; title="$(name), equation $(equation) self-field",
        xlabel="Δx", ylabel="Δz", aspect=DataAspect())
    data = stencil_matrix(static_stencil(prepared, equation), equation)
    hm = heatmap!(ax, -2:2, -2:2, data; colormap=:balance)
    Colorbar(fig[equation, column, Right()], hm)
end
display(fig)

## Polynomial patch tests

Rigid translation, rigid rotation, uniform dilation and uniform shear must have zero stress divergence in a homogeneous medium. Quadratic fields have the exact nonzero residuals listed below. These tests diagnose the LHS independently of time marching.

In [ ]:
X = repeat(reshape(Float64.(coordinates), :, 1), 1, n)
Z = repeat(reshape(Float64.(coordinates), 1, :), n, 1)
fields = (
    translation=(ones(n,n), fill(-0.4,n,n)),
    rotation=(-Z, X),
    dilation=(X, Z),
    shear=(Z, X),
    ux_x2=(X.^2, zeros(n,n)),
    ux_z2=(Z.^2, zeros(n,n)),
    ux_xz=(X.*Z, zeros(n,n)),
    uz_xz=(zeros(n,n), X.*Z),
)
expected = (
    translation=(0.0,0.0), rotation=(0.0,0.0), dilation=(0.0,0.0), shear=(0.0,0.0),
    ux_x2=(-2(λ0+2μ0),0.0), ux_z2=(-2μ0,0.0),
    ux_xz=(0.0,-(λ0+μ0)), uz_xz=(-(λ0+μ0),0.0),
)
function static_residual_at_center(prepared, fieldPair)
    ux, uz = fieldPair
    future = vcat(vec(ux), vec(uz))
    known = zeros(Float64, prepared.NpointsSpace, prepared.NField,
        prepared.timePointsUsedForOneStep - 1)
    for k in axes(known,3)
        known[:,1,k] .= vec(ux); known[:,2,k] .= vec(uz)
    end
    residual = prepared.A_unknown * future + prepared.L_known * vec(known)
    rows = (1 + prepared.NField*(centerLinear-1), 2 + prepared.NField*(centerLinear-1))
    (residual[rows[1]], residual[rows[2]])
end
function calibrated_patch_results(prepared)
    raw = map(field -> static_residual_at_center(prepared, field), fields)
    # Calibrate each equation-row normalization with its pure quadratic test.
    sx = expected.ux_x2[1] / raw.ux_x2[1]
    sz = expected.ux_z2[1] / raw.ux_z2[1]
    # sx and sz should agree if λ and μ contributions share one row normalization.
    rowScale = sx
    calibrated = map(v -> rowScale .* collect(v), raw)
    errors = map((value, target) -> norm(value .- collect(target)), calibrated, expected)
    (; raw, calibrated, errors, scale_from_ux_x2=sx, scale_from_ux_z2=sz,
       lambda_mu_scale_mismatch=abs(sx-sz)/max(abs(sx),abs(sz),eps()))
end
patchTests = map(calibrated_patch_results, methods)
summary = map(result -> (maximum_error=maximum(values(result.errors)),
    lambda_mu_scale_mismatch=result.lambda_mu_scale_mismatch,
    errors=result.errors), patchTests)
display(summary)

mixedDerivativeDiagnosis = map(patchTests) do result
    (
        ux_equals_xz=(observed_equation_z=result.calibrated.ux_xz[2],
            expected=-(λ0+μ0), missing=result.calibrated.ux_xz[2] + λ0 + μ0),
        uz_equals_xz=(observed_equation_x=result.calibrated.uz_xz[1],
            expected=-(λ0+μ0), missing=result.calibrated.uz_xz[1] + λ0 + μ0),
    )
end
display(mixedDerivativeDiagnosis)
strictTolerance = 1e-9
backendPatchTolerance = 500 * eps(Float32)
backendMaterialTolerance = 20 * eps(Float32)
coefficientHealthGate = map(summary) do result
    (patch_error=result.maximum_error,
     lambda_mu_mismatch=result.lambda_mu_scale_mismatch,
     strict_passed=result.maximum_error <= strictTolerance &&
         result.lambda_mu_scale_mismatch <= strictTolerance,
     backend_precision_passed=
         result.maximum_error <= backendPatchTolerance &&
         result.lambda_mu_scale_mismatch <= backendMaterialTolerance,
     passed=result.maximum_error <= backendPatchTolerance &&
         result.lambda_mu_scale_mismatch <= backendMaterialTolerance)
end
display(coefficientHealthGate)
@assert all(result.passed for result in values(coefficientHealthGate))
coefficientGatePath = joinpath(flexopt_root, "data",
    "elastic_lhs_coefficient_gate.jld2")
jldsave(coefficientGatePath; coefficient_health=coefficientHealthGate,
    taylor_inverse_mode=:hierarchical_constrained,
    supplementary_order=2, tested_points=(3, 4, 5),
    precision=:Float32_GPU, strict_tolerance=strictTolerance,
    backend_patch_tolerance=backendPatchTolerance,
    backend_material_tolerance=backendMaterialTolerance,
    generated_at=string(Dates.now()))
@show coefficientGatePath


## Interpretation

- If tensor and explicit OPT disagree, the tensor declaration/`@tullio` path is implicated.
- If both OPT forms agree but fail the patch tests, inspect equation (54), especially its material-point/field-point axes and derivative multi-indices.
- If `convFD3` matches centered FD but OPT3 does not, the issue is specific to the OPT test/interpolation construction rather than the elastic PDE.
- The hierarchical Taylor inverse preserves the complete total-degree-two subspace exactly, including mixed derivatives, while supplementary conditions act only in its null space.
- OPT3, OPT4, OPT5 and the `supplementaryOrder=0` control must all pass the same polynomial tests before wave propagation is enabled.
- The report keeps both a strict `1e-9` Float64 verdict and a backend-aware Float32 GPU verdict. OPT4/OPT5 may fail the former while remaining consistent at Metal precision.
- `coefficientHealthGate` is the executable pre-propagation criterion; a backend-precision failure blocks the benchmark rather than producing misleading waveforms.